# 05 — Experiments

**Workstream E, task E1.** Runs all eight strategies on all 100 experiments and builds the **raw results
frame**: one row per experiment × strategy, every metric before costs, after costs, and after 2× costs.
Notebook 06 summarises this frame, and notebook 07 re-cuts it for robustness.

| Task | Output |
| --- | --- |
| E1 | `results`: 800 rows (100 experiments × 8 strategies), columns (basis, metric) with basis gross / net / net_2x |
| E10 | The optimisation-failure log, Table A.1's source |

Nothing is written to disk. The grid takes about half a minute at the 10-asset subset size, so notebooks
06 and 07 rebuild it from the seed in the same way, and this notebook shows that the rebuild is exact.

Engine: `src/backtest.py` (notebook 03). Strategies: `src/strategies.py` (notebook 04). Metrics:
`src/performance.py`, defined in Table 4.1 (notebook 06).

## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import backtest as bt
from src import config as cfg
from src import data, fmt, viz
from src import performance as pf
from src import strategies as st

viz.use_house_style()
np.random.seed(cfg.SEED)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 140)

---

## E1 — The grid

Every strategy runs on the same 100 (window, subset) pairs, under the same engine, constraints and cost
model. The 2× cost runs make the same trades as the 1× runs and differ only in the costs deducted, so
the 2× column is ready for workstream F without a second design. Each weight decision is computed
once and replayed for the 2× run (`performance._replay`), so the optimisers run once per rebalance.

In [2]:
close, volume, extraction_date = data.download_raw()
panel = data.build_panel(close, volume, extraction_date)
prices = panel.prices[cfg.ASSET_ORDER]
experiments = bt.sample_experiments(prices)

grid = pf.run_grid(prices, experiments, st.STRATEGIES, panel.risk_free)
results = pf.results_frame(grid, panel.risk_free)

print(f"Extraction date : {panel.extraction_date.isoformat()}")
print(f"Seed            : {cfg.SEED}")
print(f"Runs            : {len(grid.runs):,} at 1x costs + {len(grid.stressed):,} at 2x")

Extraction date : 2026-10-07
Seed            : 20261001
Runs            : 800 at 1x costs + 800 at 2x


---

## The raw results frame

Values are decimals (ratios unitless). Percent conversion happens only when a table is displayed.

In [3]:
print(f"results: {results.shape[0]} rows x {results.shape[1]} columns")
print("Bases  :", list(results.columns.levels[0]))
print("Metrics:", list(results.columns.levels[1]))
display(results["net"].head(16).round(4))

results: 800 rows x 24 columns
Bases  : ['gross', 'net', 'net_2x']
Metrics: ['ann_return', 'ann_vol', 'calmar', 'max_dd', 'sharpe', 'sortino', 'sterling', 'turnover']


metric               ann_return  ann_vol  sharpe  sortino  max_dd  calmar  sterling  turnover
experiment strategy                                                                          
1          EW            0.0911   0.1297  0.5728   0.8225 -0.1415  0.6436    0.3771    0.2959
           GMV           0.0224   0.0720  0.0480   0.0673 -0.0464  0.4817    0.1527    0.7398
           MV            0.0137   0.3316  0.1422   0.1991 -0.3410  0.0401    0.0310    2.1067
           MSR           0.0298   0.2299  0.1497   0.2112 -0.2382  0.1251    0.0881    2.6410
           IV            0.0860   0.1080  0.6212   0.8998 -0.1044  0.8236    0.4208    0.3601
           ERC           0.0635   0.0992  0.4555   0.6560 -0.0921  0.6888    0.3303    0.2907
           MDP           0.0365   0.0864  0.2118   0.3023 -0.0742  0.4915    0.2093    0.6360
           MDC           0.0549   0.1131  0.3410   0.4906 -0.1218  0.4505    0.2474    0.8264
2          EW            0.0546   0.1202  0.3241   0.4545 -0.1281  0.4264    0.2395    0.2474
           GMV           0.0357   0.0724  0.2243   0.3238 -0.0495  0.7214    0.2387    0.6541
           MV           -0.1930   0.2604 -0.7744  -0.9806 -0.3736 -0.5166   -0.4076    5.0545
           MSR          -0.0873   0.2045 -0.4491  -0.5939 -0.2613 -0.3342   -0.2417    2.4832
           IV            0.0498   0.1044  0.3119   0.4403 -0.0997  0.4991    0.2492    0.3377
           ERC           0.0347   0.0968  0.1788   0.2520 -0.0883  0.3926    0.1841    0.3069
           MDP           0.0110   0.0834 -0.0840  -0.1165 -0.0920  0.1199    0.0574    0.7357
           MDC           0.0235   0.1016  0.0686   0.0954 -0.1160  0.2029    0.1090    0.8131

In [4]:
# Integrity of the frame, asserted.
# 1. Complete: every experiment x strategy, nothing dropped, nothing missing.
assert len(results) == cfg.N_EXPERIMENTS * len(cfg.STRATEGY_ORDER)
assert results.notna().all().all(), results.columns[results.isna().any()].tolist()

# 2. Every run is a full evaluation year.
assert all(len(r.net) == cfg.TRADING_DAYS for r in grid.runs.values())

# 3. Costs only ever subtract, and 2x subtracts more, in every single run.
assert (results[("net", "ann_return")] <= results[("gross", "ann_return")] + 1e-15).all()
assert (results[("net_2x", "ann_return")] <= results[("net", "ann_return")] + 1e-15).all()

# 4. Turnover is a property of the trades, not of the cost level.
assert all(grid.runs[k].turnover == grid.stressed[k].turnover for k in grid.runs)

# 5. Reproducible: rebuilding from the seed gives the identical frame, to the last bit.
rebuilt = pf.results_frame(pf.run_grid(prices, bt.sample_experiments(prices), st.STRATEGIES, panel.risk_free),
                           panel.risk_free)
pd.testing.assert_frame_equal(rebuilt, results, check_exact=True)

print("Raw results frame: complete, consistent across cost bases, and reproducible under the seed.")

Raw results frame: complete, consistent across cost bases, and reproducible under the seed.


---

## E10 — Optimisation failures

Every failure from every run, 1× and 2× (which fail identically, because costs never affect the weights).
The appendix table uses the 1× runs.

In [5]:
failures = bt.failure_log(list(grid.runs.values()))
failures_2x = bt.failure_log(list(grid.stressed.values()))
assert failures.equals(failures_2x)

counts = failures.groupby("strategy").size().reindex(cfg.STRATEGY_ORDER, fill_value=0)
print(f"Failures: {len(failures)} of {len(grid.runs) * len(bt.Schedule().execution_positions()):,} "
      "rebalances (1x runs). By strategy:", counts.to_dict())
if failures.empty:
    print("Failure log: empty. No optimisation failed in any of the 800 runs; Table A.1 will say so.")
else:
    display(failures.assign(date=pd.to_datetime(failures["date"]).dt.date))

Failures: 0 of 3,200 rebalances (1x runs). By strategy: {'EW': 0, 'GMV': 0, 'MV': 0, 'MSR': 0, 'IV': 0, 'ERC': 0, 'MDP': 0, 'MDC': 0}
Failure log: empty. No optimisation failed in any of the 800 runs; Table A.1 will say so.


---

## Handoff

```python
grid = pf.run_grid(prices, experiments, st.STRATEGIES, panel.risk_free)
results = pf.results_frame(grid, panel.risk_free)     # identical in 06 and 07
```

| Object | Used by |
| --- | --- |
| `results[("net", ...)]` | 06: Tables 4.2–4.3, Figures 4.1–4.2 |
| `results[("gross", ...)]` | 06: Table 4.4, appendix Table A4.1 |
| `results[("net_2x", ...)]` | 07: Table 5.1 |
| `grid.runs[(experiment, strategy)]` | 06: Figures 4.3–4.4 (daily wealth and weights); 07: concentration |
| `failures` | 06: Table A.1 |